# Reproducing Johnson et al. (2018) from CRCNS fcx-2

**Paper.** Johnson EL, Adams JN, Solbakk A-K, Endestad T, Larsson PG, Ivanovic J, Meling TR, Lin JJ,
Knight RT (2018). Dynamic frontotemporal systems process space and time in working memory. *PLoS
Biology* 16, e2004274. [doi:10.1371/journal.pbio.2004274](https://doi.org/10.1371/journal.pbio.2004274).
The claims file was written from the version of record (PMC5895055, full text with Methods and
figure legends; the supplementary tables were not supplied).

**Data set.** CRCNS [fcx-2](https://crcns.org/data-sets/pfc/fcx-2),
[doi:10.6080/K0VX0DQD](https://doi.org/10.6080/K0VX0DQD), read from the public bucket
`crcnsarchive` under `fcx-2/live/`. CRCNS data sets are not versioned, so the version is recorded
as `unversioned`. The paper cites the data set by its DOI and no version. The bucket holds the live
files and, under `fcx-2/orig/ver_0_6/`, only the documents and scripts of version 0.6; the data
files exist once, under `live/`, which is the published version 0.61 (descriptor of April 4, 2018,
whose change from 0.6 is an updated `psianalysis.m`). This is the only version of the data, so it is
pinned. MD5 sums of the files read, from the data set's `checksums.md5`:

| file | md5 |
|---|---|
| data_indiv/s1.tar.gz | 0bbbcce3cc2738bc7ce5545b27843d4c |
| data_indiv/s2.tar.gz | 5ead31d89d1ae4a50adae20054bb6572 |
| data_indiv/s3.tar.gz | 740991d34257dabc1fdd8a467f1e80a2 |
| data_indiv/s4.tar.gz | 306d554f9e9ec77a35b68879f301f6b1 |
| data_indiv/s5.tar.gz | 66673b1504648aff5ea1f2e0bea6fcdd |
| data_indiv/s6.tar.gz | 68608a2237469bc285b928298b1e8f3f |
| data_indiv/s7.tar.gz | 639d8e97e026f9b8dea08610c51af00e |
| data_indiv/s8.tar.gz | 6aa467667c0fec9566f49fd19c8461e8 |
| data_indiv/s9.tar.gz | bfd6d72c7663db18db4f0fe90cea63c7 |
| data_indiv/s10.tar.gz | 5eb078815470c8d1b70b06d8f2be5300 |
| docs/summary_channel_analysis.csv | 79adb25db44e195ad952359ed43d9a3a |
| docs/summary_file.csv | 6d214de444bf222cee1c3291b6191a6f |
| analysis_scripts.tar.gz | 8b5b07094a3a31e8bb4828afcdffd2f2 (read for the methods, not run) |

**What the data set holds.** For each of the 10 subjects, the preprocessed bipolar data
(`data_derived.mat`: segments `pre`, `encmain` and `proc`, each 4.5 s long with a time axis from -1.5
to 3 s, and a trial table per segment), the filtered monopolar data, a per-trial table and electrode
coordinates; the analysed channels per region and hemisphere (`summary_channel_analysis.csv`); a
behaviour table per subject; and the MATLAB analysis scripts. The tar files are streamed and read
in memory; only the derived data of the 254 analysed channels are kept.

**Methods.** Everything not in the claims file is taken from the paper's Methods or from the
analysis code deposited with the data, as the helper module `fcx2.py` says parameter by parameter.

Every window is placed by the trial table and the descriptor (the main reading), 1.5 s earlier
on the derived time axes than the deposited code places it; the data check below shows why. ERPs,
power, PLV and both PSI measures are also run with the code's windows.

| claim | tested | rests on an assumption |
|---|---|---|
| behavior-proficient | yes | |
| behavior-no-difficulty-difference | yes | numeric coding of condition; response time from summary_file.csv; accuracy model on subject-level logits |
| electrode-counts | yes | |
| correct-trial-counts | yes | |
| erp-no-condition-difference | yes | baseline as erp_compute.m subtracts it (trial mean) |
| power-theta-sustained | yes | |
| power-condition-exclusions | yes | |
| plv-theta-peak | yes | |
| psi-theta-shift | yes | window placement of the trial table |
| crossspectral-shift | yes | |
| phase-directing-counts | yes | |
| crossspectral-equal-strength | blocked (methods_underspecified) | |
| theta-peak-bands | yes | |
| pac-* (8 claims) | blocked (methods_underspecified) | |


In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy import stats

import fcx2

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 400)

started = time.time()
chtab = fcx2.read_channel_table()
summary = fcx2.read_summary_file()
R = {}

In [ ]:
# one subject at a time: stream, check, run every per-subject measure, keep only the results
for sid in ['s1', 's2', 's3', 's4']:
    R[sid] = fcx2.process_subject(sid, chtab)
print(f"{len(R)} subjects done after {(time.time() - started) / 60:.1f} min")

In [ ]:
# one subject at a time: stream, check, run every per-subject measure, keep only the results
for sid in ['s5', 's6']:
    R[sid] = fcx2.process_subject(sid, chtab)
print(f"{len(R)} subjects done after {(time.time() - started) / 60:.1f} min")

In [ ]:
# one subject at a time: stream, check, run every per-subject measure, keep only the results
for sid in ['s7', 's8', 's9']:
    R[sid] = fcx2.process_subject(sid, chtab)
print(f"{len(R)} subjects done after {(time.time() - started) / 60:.1f} min")

In [ ]:
# one subject at a time: stream, check, run every per-subject measure, keep only the results
for sid in ['s10']:
    R[sid] = fcx2.process_subject(sid, chtab)
print(f"{len(R)} subjects done after {(time.time() - started) / 60:.1f} min")

## Data check

In [ ]:
rows = []
for sid, r in R.items():
    s = r["sanity"]
    ti = r["trialinfo"]
    rows.append({
        "subject": sid, "fs": s["fs"], "hdr_fs": sorted(set(s["hdr_fs"].values())),
        "derived_labels": s["n_labels"], "analysed": len(r["chans"]),
        "trials_table": len(s["trials"]), "trials_derived": s["n_trials_derived"],
        "correct_derived": int((ti[:, 1] == 1).sum()),
        "samples_pre": s["n_samples"]["pre"], "samples_enc": s["n_samples"]["encmain"],
        "samples_proc": s["n_samples"]["proc"],
        "time_enc": s["time_range"]["encmain"][:2], "same_start": all(s["same_start"].values()),
        "x_min_uV": round(s["x_range"][0], 1), "x_max_uV": round(s["x_range"][1], 1),
        "load_s": round(s["load_s"]),
    })
info = pd.DataFrame(rows)
print(info.to_string(index=False))
print("\nmembers of each tar file:")
for sid, r in R.items():
    print(sid, {k.split('/')[-1]: v for k, v in r["sanity"]["members"].items()})
print("\nAll analysed channels of docs/summary_channel_analysis.csv found among the derived labels: yes "
      "(load_subject raises otherwise)")
print("Channels per region in the channel table:", chtab.region.value_counts().to_dict(), "total", len(chtab))

The descriptor says the derived structures have `fsample: 1000`; s8 and s9 are at 512 Hz, as its
note on the trial table says and as the paper says for the Oslo subjects. Trials differ in length by at
most a sample or two within a segment; the analysis keeps the samples all trials share, from the
common first time (-1.5 s).

In [ ]:
rows = []
for sid, r in R.items():
    s = r["sanity"]
    rows.append({"subject": sid, **{k: np.round(v, 2).tolist()[:4] + (["..."] if len(v) > 4 else []) for k, v in s["offsets"].items()},
                 "overlap_maxdiff": s["overlap_maxdiff"], "overlap_samples": s["overlap_samples"],
                 "trialinfo_same_in_segments": s["trialinfo_same_segments"],
                 "trialinfo_equals_trial_table": s["trialinfo_matches_table"]})
print(pd.DataFrame(rows).to_string(index=False))

The three derived segments of a trial are cut from one continuous recording: placed by their
`sampleinfo`, their overlapping samples are identical (largest difference 0) in every subject. In
nine subjects the first sample of `encmain` is the trial table's column H ("encoding and pre-cue
delay epoch start"), the first sample of `proc` is 0.3 s before column I ("post-cue delay epoch
start") and the first sample of `pre` is about 0.1 s after column G ("pretrial epoch start, 1000 ms
before trial start"; 98 to 108 samples at 1 kHz, 51 at 512 Hz). In s10 the offsets are not whole samples and vary over trials (the table's columns
for s10 are not on the derived data's sample grid; for two trials `pre` is 114 samples after G), so
for s10 the table does not place the derived segments exactly. In every subject `trialinfo` is the
same in the three segments and equals the trial table's columns, except s10, where the
trial-table check fails as printed above.

This bears on where time 0 is. Each derived segment runs from -1.5 s to 3.0 s, and its first sample
(-1.5 s) is the table's epoch start. The analysis code takes encoding as [0, 1.5] s of `encmain`, the
postcue delay as [0.3, 1.2] s of `proc` and the baseline as [0.65, 0.85] s of `pre` ("-250 to 50 ms
pretrial"), which places stimulus onset at 0 s, 1.5 s after the table's column H. The descriptor
places stimulus onset at column H itself, 1.5 s earlier, and the table's columns agree with the
task's timing (H minus G is 1.9 s, the 1 s fixation, 800 ms start screen and 100 ms fixation; I
minus H is 600 ms plus the delay plus the 800 ms test cue). While preparing the first run, a
derived trial of s8 was compared with the monopolar data in `data_primary.mat` (not run here, as it
streams the primary files): the bipolar `CL_02` and `BR_03` traces equal the difference of
adjacent contacts from sample H on, so `sampleinfo` indexes the primary data as the table does.
The check below decides between the two placements from the data.

In [ ]:
edges = R["s1"]["sanity"]["evoked_edges"]
centres = edges[:-1] + 0.025
for seg in ("pre", "encmain", "proc"):
    hg9 = np.nanmean([r["sanity"]["hg"][(seg, 900)] for r in R.values()], axis=0)
    hg11 = np.nanmean([r["sanity"]["hg"][(seg, 1150)] for r in R.values()], axis=0)
    ev = np.nanmean([r["sanity"]["evoked_rms"][seg] / np.nanmean(r["sanity"]["evoked_rms"][seg]) for r in R.values()], axis=0)
    print(f"{seg}: high-gamma amplitude relative to its mean, 900 ms delay trials, 100 ms steps from -1.5 s:")
    print("  ", " ".join(f"{v:.2f}" for v in hg9[::2]))
    print("   1150 ms delay trials:")
    print("  ", " ".join(f"{v:.2f}" for v in hg11[::2]))
    print("   evoked RMS over channels relative to its mean:")
    print("  ", " ".join(f"{v:.2f}" for v in ev[::2]))
    print(f"   range of the high-gamma curves {np.nanmin([hg9, hg11]):.3f} to {np.nanmax([hg9, hg11]):.3f}; "
          f"largest 900 vs 1150 difference {np.nanmax(np.abs(hg9 - hg11)):.3f} at {centres[np.nanargmax(np.abs(hg9 - hg11))]:.2f} s")
WINS = np.arange(-1.5, 3.0, 0.25)
# per electrode: t of the 900 against the 1150 ms trials in 50 ms bins (fcx2.sanity_subject),
# pooled over subjects: the fraction of electrode bins with |t| > 1.96 per 250 ms window, and its p
# against 200 permutations of the jitter labels (the same permutation index summed over subjects)
print("\nPer electrode, 900 against 1150 ms delay trials, in 250 ms windows from -1.5 s")
FIRST = {"pre": ("test cue onset", 1.8, None), "encmain": ("test cue onset", 0.0, 1.5),
         "proc": ("response screen", -0.3, 1.2)}
for seg in ("pre", "encmain", "proc"):
    ev, t_tab, t_code = FIRST[seg]
    print(f"{seg}: first event that depends on the delay, {ev}: {t_tab:+.2f} s by the trial table, "
          + (f"{t_code:+.2f} s by the code's windows" if t_code is not None else "after the segment's end by the code's windows"))
    for name in ("hg", "erp"):
        obs = sum((np.abs(r["sanity"]["jitter_t"][(seg, name)]) > 1.96).sum(axis=0) for r in R.values())
        null = sum(r["sanity"]["jitter_t"][(seg, name, "null")].astype(int) for r in R.values())
        nel = sum(r["sanity"]["jitter_t"][(seg, name)].shape[0] for r in R.values())
        frac, pv = [], []
        for w0 in WINS:
            m = (centres >= w0) & (centres < w0 + 0.25)
            frac.append(obs[m].sum() / (nel * m.sum()))
            pv.append((np.sum(null[:, m].sum(axis=1) >= obs[m].sum()) + 1) / (null.shape[0] + 1))
        print(f"  {name:4s} frac " + " ".join(f"{v:.2f}" for v in frac))
        print(f"  {'':4s} p    " + " ".join(f"{v:.2f}" for v in pv))
print("  windows   " + " ".join(f"{w:+.2f}" for w in WINS))
# the electrode with the largest high-gamma modulation in encmain, per subject
rows = []
for sid, r in R.items():
    hm = r["sanity"]["jitter_t"][("encmain", "hg_mean")][:, 1:]          # first bin is the segment edge
    rng_ = hm.max(axis=1) - hm.min(axis=1)
    e = int(np.argmax(rng_))
    rows.append({"subject": sid, "electrode": r["chans"].label[e], "region": r["chans"].region[e],
                 "log_amp_range": round(float(rng_[e]), 3), "peak_at_s": round(float(centres[1:][np.argmax(hm[e])]), 2),
                 "trough_at_s": round(float(centres[1:][np.argmin(hm[e])]), 2),
                 "curve_100ms": np.round(hm[e][::2], 2).tolist()})
print(pd.DataFrame(rows).to_string(index=False))


Pooled over all channels, the high-gamma amplitude (70 to 150 Hz) and the evoked response show no
task-locked modulation large enough to place the events (the curves stay within a few percent of
their mean apart from the first bin, the edge of the segment). The per-electrode test of the 900
against the 1150 ms delay trials does place them. The two groups can only differ from the first
event whose time depends on the delay. In `proc` the ERPs part from -0.25 s to 0.75 s (fraction of
electrode bins with |t| > 1.96 of 0.07 to 0.12, p from 0.05 down to 0.005, the smallest 200
permutations can give),
which is where the trial table puts the response screen (-0.3 s), and not at 1.2 s, where the
code's windows put it (p 0.29 to 0.94 in the windows from 1.0 to 2.0 s); high gamma parts at -0.25 to 0.25 s (p
0.01 to 0.05). In `encmain` the ERPs part from 0.25 s (p 0.01 at 0.25, 0.5 and 1.0 s), after the
test cue at 0 s by the table and before 1.5 s, where the code puts it. In `pre` they part at 2.25
to 2.5 s (p 0.01 and 0.03), after the test cue at 1.8 s by the table; by the code's windows `pre`
holds no event that depends on the delay. So the data are consistent with the trial table's
placement and not with the code's, and every window below is placed by the table: encoding
[-1.5, 0] s and the precue delay [-0.9, 0] s of `encmain`, the postcue delay [-1.2, -0.3] s of
`proc` and the baseline [-0.85, -0.65] s of `pre`. The encoding window then starts at the first
sample of `encmain`, so its first few hundred ms carry the filters' edge effects. ERPs, power, PLV
and both PSI measures are also computed with the code's windows. The electrode with the largest
high-gamma range per subject is printed for reference; in 9 of 10 subjects its peak is the last bin
(2.98 s), the segment's edge.

In [ ]:
# the deposited analysis scripts, read to quote the lines the windows, the ERP baseline and the
# PSI are taken from (not run)
SCRIPTS = fcx2.read_scripts()


def quote(name, first, last):
    lines = SCRIPTS[name].splitlines()
    print(f"--- {name}, lines {first}-{last}")
    for k in range(first - 1, last):
        print(f"{k + 1:4d}  {lines[k]}")


print("files:", sorted(SCRIPTS))
quote("erp_compute.m", 44, 47)        # baseline window
quote("erp_compute.m", 74, 97)        # baseline correction and segment latencies
for name in ("power_compute.m", "psi_compute.m", "cspsi_compute.m", "plv_compute.m", "pac_compute.m", "dirpac_compute.m"):
    for k, line in enumerate(SCRIPTS[name].splitlines()):
        if "latency" in line:
            print(f"{name}:{k + 1}  {line.strip()}")
quote("psi_compute.m", 29, 36)        # spectral step
quote("psianalysis.m", 26, 30)        # cross-spectral density, no conjugate
quote("psianalysis.m", 46, 72)        # per-trial PSI, trial mean, frequency-shuffled null


`erp_compute.m` (lines 84 to 88) subtracts `nanmean(pretrial.trial)`, the mean over trials of each
trial's baseline mean, so one value per electrode is subtracted from every trial; the ERP section
follows it, and subtracting each trial's own baseline mean is run as a variant. Every script cuts
the same latencies from the derived segments ([0 1.5] of `encmain`, [0.6 1.5] for the precue
delay, [0.3 1.2] of `proc`, [0.65 0.85] of `pre`, commented "-250 to 50 ms pretrial"). The PSI
script uses FieldTrip `mtmfft` with a Hanning taper and `t_ftimwin` set, which `mtmfft` ignores, so
the spectrum comes from one window over the 900 ms segment. `psianalysis.m` multiplies the Fourier
coefficients without a conjugate, takes the trial mean of per-trial PSI as the observed value, and
z-scores it against PSI computed from the trial-mean normalised cross-spectrum with its frequencies
shuffled.


In [ ]:
# behaviour tables: the trial table against docs/summary_file.csv
rows = []
for sid, r in R.items():
    t = r["sanity"]["trials"]
    row = {"subject": sid, "n_trials": len(t), "per_condition": t.condition.value_counts().sort_index().tolist(),
           "jitters": sorted(t.jitter.unique().tolist())}
    srow = summary.set_index("subject").loc[sid]
    for c, name in ((1, "identity"), (2, "spatial"), (3, "temporal")):
        tc = t[t.condition == c]
        row[f"acc_{name}"] = round(tc.accuracy.mean(), 3)
        row[f"acc_{name}_doc"] = srow[f"acc_{name}"]
        row[f"rt_{name}"] = round(tc[tc.accuracy == 1].rt.mean(), 1)
        row[f"rt_{name}_doc"] = round(srow[f"rt_{name}"], 1)
    rows.append(row)
beh = pd.DataFrame(rows)
print(beh.to_string(index=False))
acc_ok = all(np.isclose(beh[f"acc_{n}"], beh[f"acc_{n}_doc"], atol=1e-3).all() for n in ("identity", "spatial", "temporal"))
rt_ok = all(np.isclose(beh[f"rt_{n}"], beh[f"rt_{n}_doc"], atol=0.5).all() for n in ("identity", "spatial", "temporal"))
print("\naccuracy in summary_file.csv equals the trial tables (all 120 trials):", acc_ok)
print("response time in summary_file.csv equals the mean over correct trials of the trial tables:", rt_ok)
print("subjects' age and sex:", summary[["subject", "age", "sex"]].values.tolist())
print(f"age mean {summary.age.mean():.1f} SD {summary.age.std():.1f} range {summary.age.min()}-{summary.age.max()}; males {(summary.sex == 'M').sum()}")
# recordings deposited more than once
sig = {sid: (tuple(r["sanity"]["trials"].rt.tolist()), tuple(np.round(r["erp_mean"]["enc"][0, :50], 4))) for sid, r in R.items()}
dups = [(a, b) for i, a in enumerate(sig) for b in list(sig)[i + 1:] if sig[a][0] == sig[b][0] or sig[a][1] == sig[b][1]]
print("subjects with identical response-time sequences or identical evoked traces:", dups or "none")
# where do the response times of summary_file.csv come from? Candidate summaries of the trial
# tables, per subject and condition, against the file (a match is within 0.05 ms)
cands = {}
for sid, r in R.items():
    t = r["sanity"]["trials"]
    ti = r["trialinfo"]
    k = pd.DataFrame({"accuracy": ti[:, 1], "condition": ti[:, 3], "rt": ti[:, 5], "trial": ti[:, 0]})
    for c in (1, 2, 3):
        tc, kc = t[t.condition == c], k[k.condition == c]
        cr = tc[tc.accuracy == 1]
        cands[(sid, c)] = {"mean correct": cr.rt.mean(), "median correct": cr.rt.median(), "mean all": tc.rt.mean(),
                           "median all": tc.rt.median(), "mean incorrect": tc[tc.accuracy == 0].rt.mean(),
                           "mean kept correct": kc[kc.accuracy == 1].rt.mean(), "mean kept all": kc.rt.mean(),
                           "median kept correct": kc[kc.accuracy == 1].rt.median(),
                           "geometric mean correct": np.exp(np.log(cr.rt).mean()),
                           "mean correct, by trial type": t[(t.accuracy == 1) & (t.trial_type == (1 if c == 1 else 2))].rt.mean()}
    cr = t[t.accuracy == 1]
    for b in range(3):
        cands[(sid, f"block{b + 1}")] = {"mean correct, trials of block": cr[(cr.trial > 40 * b) & (cr.trial <= 40 * (b + 1))].rt.mean()}
doc = summary.set_index("subject")
rows = []
for sid in R:
    for c, name in ((1, "identity"), (2, "spatial"), (3, "temporal")):
        v = doc.loc[sid, f"rt_{name}"]
        hits = [f"{s2}/{c2}: {k}" for (s2, c2), d in cands.items() for k, x in d.items() if abs(x - v) < 0.05]
        rows.append({"subject": sid, "condition": name, "summary_file": round(v, 1),
                     "mean_correct": round(cands[(sid, c)]["mean correct"], 1),
                     "median_correct": round(cands[(sid, c)]["median correct"], 1),
                     "mean_all": round(cands[(sid, c)]["mean all"], 1), "matches": hits or "none"})
prov = pd.DataFrame(rows)
print("\nsummary_file.csv response times against summaries of the trial tables (any subject, condition or block):")
print(prov.to_string(index=False))
print("subject-condition values matched by some summary:", int((prov.matches != "none").sum()), "of", len(prov),
      "; subjects with all three matched:", sorted(prov.groupby("subject").matches.apply(lambda s: (s != "none").all()).loc[lambda s: s].index.tolist()))


The accuracies of `docs/summary_file.csv` equal those of the per-trial tables over all 120 trials,
as its descriptor says. Its response times do not equal the mean over correct trials of the
per-trial tables in 7 of 10 subjects (they do in s6, s8 and s9; for s1, identity, 1948 against 2348
ms), although the descriptor says they are "calculated on all correct trials"; the paper's
condition means (1659, 1335, 1475 ms) are those of `summary_file.csv`. No other summary of the
trial tables tried here gives them either (medians, all trials, kept trials, other subjects'
rows, blocks of 40 trials, trial type): 9 of 30 values are matched, all by the mean over correct
trials of s6, s8 and s9. Age and sex match the paper's description (37 ± 13 years, 22 to 69, 7
males). No subject's data appear twice.

## behavior-proficient

> All ten subjects performed the task well above chance, with accuracy from 0.79 to 0.97 (chance
> 0.5).

In [ ]:
acc_all = {sid: r["sanity"]["trials"].accuracy.mean() for sid, r in R.items()}
acc_kept = {sid: (r["trialinfo"][:, 1] == 1).mean() for sid, r in R.items()}
tab = pd.DataFrame({"all_120_trials": acc_all, "kept_trials": acc_kept}).round(3)
print(tab.to_string())
for col in tab:
    v = tab[col]
    print(f"{col}: min {v.min():.3f} (expected 0.79 +- 0.079), max {v.max():.3f} (expected 0.97 +- 0.097), "
          f"all in [0.711, 1.0]: {bool(((v >= 0.711) & (v <= 1.0)).all())}; "
          f"min ok {abs(v.min() - 0.79) <= 0.079}, max ok {abs(v.max() - 0.97) <= 0.097}")

**Result: reproduced.** Over all 120 trials (reading a) accuracy runs from 0.792 to 0.967, every
subject inside 0.711 to 1.0, both bounds within tolerance of 0.79 and 0.97. Over kept trials
(reading b) it runs from 0.792 to 1.000, also within tolerance.

## behavior-no-difficulty-difference

> Accuracy and correct-trial response time did not differ between the identity, spatial and
> temporal conditions (mean accuracy 0.90, 0.92, 0.89; mean response time 1659, 1335, 1475 ms).

The models are fitted on one value per subject and condition (30 rows), as the criterion says;
accuracy enters as the empirical logit, log((k + 0.5) / (n - k + 0.5)), as some subjects are at
100% in a condition (an assumption: the paper's "logit mixed-effects model" is not specified
further). The criterion judges reading (c), condition as one numeric term coded identity 1,
spatial 2, temporal 3 (the coding is an assumption the criterion states), which leaves F(1,28) on
30 rows. Its F is the squared t of the slope, with p from F(1,28), the residual degrees of freedom
MATLAB's mixed-model F test uses by default (criterion). Reported, not judged: (a) each relation
condition against identity and (b) one 3-level condition term (Wald test), and a trial-level
variant of response time (all correct trials, subject random intercept).

Response time has two sources that disagree in 7 of 10 subjects: the mean over correct trials of
each trial table, and the per-subject values of `docs/summary_file.csv`, whose condition means are
the paper's. Both are tested; the outcome is judged on `summary_file.csv` as the values the paper's
statistics were computed from (an assumption). Accuracy is the same in both sources and is taken
under the two readings of behavior-proficient, (a) all 40 trials per condition and (b) the trials
kept in the derived data.

In [ ]:
def behaviour_rows(kept):
    rows = []
    for sid, r in R.items():
        if kept:
            ti = r["trialinfo"]
            t = pd.DataFrame({"accuracy": ti[:, 1], "condition": ti[:, 3], "rt": ti[:, 5]})
        else:
            t = r["sanity"]["trials"]
        for c in (1, 2, 3):
            tc = t[t.condition == c]
            k, n = tc.accuracy.sum(), len(tc)
            rows.append({"subject": sid, "cond": c, "acc": k / n, "k": k, "n": n,
                         "elogit": np.log((k + 0.5) / (n - k + 0.5)), "rt": tc[tc.accuracy == 1].rt.mean()})
    return pd.DataFrame(rows)


def lmm_tests(d, y):
    out = {}
    m = smf.mixedlm(f"{y} ~ C(cond)", d, groups=d["subject"]).fit(reml=True)
    out["spatial_vs_identity_p"] = m.pvalues["C(cond)[T.2]"]
    out["temporal_vs_identity_p"] = m.pvalues["C(cond)[T.3]"]
    w = m.wald_test("C(cond)[T.2] = 0, C(cond)[T.3] = 0", scalar=True)
    out["three_level_chi2"], out["three_level_p"] = float(w.statistic), float(w.pvalue)
    mn = smf.mixedlm(f"{y} ~ cond", d, groups=d["subject"]).fit(reml=True)
    out["numeric_slope"], out["numeric_z"], out["numeric_p"] = mn.params["cond"], mn.tvalues["cond"], mn.pvalues["cond"]
    # reading (c), judged: F(1,28) = t^2 of the slope, p from F(1,28) (criterion)
    out["numeric_F"] = mn.tvalues["cond"] ** 2
    out["numeric_p_F128"] = stats.f.sf(out["numeric_F"], 1, 28)
    return out


for kept in (False, True):
    d = behaviour_rows(kept)
    label = "kept trials (b)" if kept else "all 120 trials (a)"
    means = d.groupby("cond")[["acc", "rt"]].agg(["mean", "std"]).round(3)
    print(f"--- {label}: mean and SD over subjects\n{means.to_string()}")
    for y in ("elogit", "rt"):
        print(f"  {y}:", {k: round(v, 4) for k, v in lmm_tests(d, y).items()})
rows = []
for sid, r in R.items():
    t = r["sanity"]["trials"]
    t = t[t.accuracy == 1]
    rows += [{"subject": sid, "cond": c, "rt": v} for c, v in zip(t.condition, t.rt)]
dt = pd.DataFrame(rows)
mt = smf.mixedlm("rt ~ C(cond)", dt, groups=dt["subject"]).fit()
w = mt.wald_test("C(cond)[T.2] = 0, C(cond)[T.3] = 0", scalar=True)
print("trial-level response time variant: p spatial", round(mt.pvalues["C(cond)[T.2]"], 4), "p temporal",
      round(mt.pvalues["C(cond)[T.3]"], 4), "3-level p", round(float(w.pvalue), 4))
# the per-subject response times of docs/summary_file.csv, whose means are the paper's
sf = summary.melt(id_vars="subject", value_vars=["rt_identity", "rt_spatial", "rt_temporal"], var_name="c", value_name="rt")
sf["cond"] = sf.c.map({"rt_identity": 1, "rt_spatial": 2, "rt_temporal": 3})
print("summary_file.csv response time, mean and SD over subjects:", sf.groupby("cond").rt.agg(["mean", "std"]).round(1).to_dict("index"))
print("  rt (summary_file.csv):", {k: round(v, 4) for k, v in lmm_tests(sf, "rt").items()})
d = behaviour_rows(False)
exp = {1: (0.90, 0.04, 1659, 701), 2: (0.92, 0.09, 1335, 574), 3: (0.89, 0.09, 1475, 734)}
for c, (a, ta, rt, trt) in exp.items():
    ma, mr = d[d.cond == c].acc.mean(), d[d.cond == c].rt.mean()
    print(f"condition {c}: accuracy {ma:.3f} vs {a} +- {ta}: {abs(ma - a) <= ta}; rt {mr:.0f} vs {rt} +- {trt}: {abs(mr - rt) <= trt}")
# the judged reading (c): condition as one numeric term, p from F(1,28), effect at p < 0.05
judged = []
for kept in (False, True):
    d = behaviour_rows(kept)
    for y, source in (("elogit", "trial tables"), ("rt", "trial tables")):
        judged.append((y, "kept" if kept else "all", source, lmm_tests(d, y)))
judged.append(("rt", "-", "summary_file.csv", lmm_tests(sf, "rt")))
print("reading (c), condition as one numeric term (paper: accuracy F(1,28) < 0.10, p > 0.75; rt F(1,28) < 2.55, p > 0.12)")
for y, trials, source, o in judged:
    print(f"  {y:7s} trials {trials:4s} {source:17s} slope {o['numeric_slope']:9.3f}  F(1,28) {o['numeric_F']:6.2f}  "
          f"p {o['numeric_p_F128']:.4f}  effect {o['numeric_p_F128'] < 0.05}")

**Result: reproduced, resting on two assumptions.** Under reading (c), the one the criterion
judges, neither model finds a condition effect on the data whose condition means are the paper's:
accuracy gives F(1,28) = 0.16, p = 0.70 over all trials (0.44, p = 0.51 over kept trials), and the
response times of `summary_file.csv` give F(1,28) = 2.41, p = 0.13, against the paper's F < 2.55,
p > 0.12. The accuracy F is above the paper's bound of 0.10, though far from an effect. The outcome
rests on the numeric coding of condition, which the criterion states as an assumption, and on
judging response time on `summary_file.csv`, whose values none of the trial-table summaries above
reproduces for 7 of 10 subjects: with the trial tables' response times, (c) finds an effect
(F(1,28) = 12.33, p = 0.0015 over all trials; 11.48, p = 0.0021 over kept trials). The condition
means are within the paper's SDs (accuracy 0.900, 0.920, 0.890; response time 1659, 1335, 1475 ms
from `summary_file.csv` and 1744, 1237, 1292 ms from the trial tables). Under the readings reported
but not judged, response time differs by condition in both sources (summary_file.csv 3-level p =
0.0076, spatial against identity p = 0.0018), and accuracy over all trials differs between spatial
and identity (p = 0.023) though not under the 3-level term (p = 0.054).

## electrode-counts

> After bipolar rereferencing there were 254 electrodes, 36 MTL, 116 lateral PFC and 102 OFC; MTL
> and PFC were covered in the same hemisphere in 9 of 10 subjects and MTL and OFC in 7 of 10.

In [ ]:
cnt = chtab.region.value_counts()
print("electrodes per region:", cnt.to_dict(), "total", len(chtab))
per = chtab.groupby(["subject", "region", "hemi"]).size().unstack(["region", "hemi"]).fillna(0).astype(int)
print(per.to_string())
def same_hemi(a, b):
    return sum(any(((chtab.subject == s) & (chtab.region == a) & (chtab.hemi == h)).any() and
                   ((chtab.subject == s) & (chtab.region == b) & (chtab.hemi == h)).any() for h in "LR")
               for s in fcx2.SUBJECTS)
n_pfc, n_ofc = same_hemi("MTL", "PFC"), same_hemi("MTL", "OFC")
print("subjects with MTL and PFC in one hemisphere:", n_pfc, "(expected 9)")
print("subjects with MTL and OFC in one hemisphere:", n_ofc, "(expected 7)")
pr = chtab.groupby(["subject", "region"]).size().unstack().fillna(0)
for reg in ("MTL", "PFC", "OFC"):
    v = pr[reg][pr[reg] > 0]
    print(f"{reg} per subject with any: mean {pr[reg].mean():.1f} SD {pr[reg].std():.1f}; range {int(v.min())}-{int(v.max())} in {len(v)} subjects")

**Result: reproduced.** The channel table lists 36 MTL, 116 PFC and 102 OFC electrodes (254), all
present in the derived data, and MTL shares a hemisphere with PFC in 9 subjects and with OFC in 7.

## correct-trial-counts

> The final data held on average 100 correct trials per subject, 33 +- 3 identity (26 to 37), 34 +- 4
> spatial (28 to 39) and 33 +- 3 temporal (28 to 37), and 10 +- 8 incorrect trials (0 to 25).

In [ ]:
rows = []
for sid, r in R.items():
    ti = r["trialinfo"]
    c = ti[:, 1] == 1
    rows.append({"subject": sid, "correct": int(c.sum()), "identity": int((c & (ti[:, 3] == 1)).sum()),
                 "spatial": int((c & (ti[:, 3] == 2)).sum()), "temporal": int((c & (ti[:, 3] == 3)).sum()),
                 "incorrect": int((~c).sum()), "kept": len(ti)})
tc = pd.DataFrame(rows).set_index("subject")
print(tc.to_string())
print(tc.agg(["mean", "std", "min", "max"]).round(2).to_string())
checks = {"correct": (100, 10, None), "identity": (33, 3, (26, 37)), "spatial": (34, 4, (28, 39)),
          "temporal": (33, 3, (28, 37)), "incorrect": (10, 8, (0, 25))}
for k, (v, tol, rg) in checks.items():
    m = tc[k].mean()
    line = f"{k}: mean {m:.1f} vs {v} +- {tol}: {abs(m - v) <= tol}"
    if rg:
        lo, hi = rg[0] - 0.1 * rg[0], rg[1] + 0.1 * rg[1]
        line += f"; all subjects in [{lo:.1f}, {hi:.1f}]: {bool(((tc[k] >= lo) & (tc[k] <= hi)).all())}"
    print(line)

**Result: reproduced.** The kept trials hold on average 100.9 correct trials per subject (32.9
identity, 34.8 spatial, 33.2 temporal) and 10.2 incorrect, each within the paper's SD, and every
subject's count lies in the widened ranges (identity 26 to 37, spatial 28 to 39, temporal 28 to 37,
incorrect 0 to 25).

## erp-no-condition-difference

> ERPs did not differ between identity, spatial and temporal conditions in any electrode, over
> encoding, precue or postcue intervals.

The test is run per subject as `erp_compute.m` does (all of a subject's analysed electrodes in one
test, so the null is the largest cluster over the subject's electrodes), and per electrode (the
Methods' "clustered per electrode") as a variant. The baseline subtracted is the trial mean of
each trial's baseline mean, as `erp_compute.m` does it (lines 84 to 88, quoted above); each trial's
own baseline mean is a variant, and so are the code's windows.

In [ ]:
rows = []
for tag, label in (("", "main: trial-mean baseline (code)"), ("_trial", "variant: per-trial baseline"), ("_alt", "variant: code's windows")):
    for sid, r in R.items():
        for test in ("erp_enc", "erp_post"):
            d = r[test + tag]
            rows.append({"reading": label, "subject": sid, "test": test, "n_el": len(d),
                         "sig_subject_null": int((d.p_subject < 0.05).sum()),
                         "sig_electrode_null": int((d.p_electrode < 0.05).sum()), "min_p_subject": round(d.p_subject.min(), 3),
                         "min_p_electrode": round(d.p_electrode.min(), 3)})
erp = pd.DataFrame(rows)
print(erp[erp.reading.str.startswith("main")].to_string(index=False))
n_el = len(chtab)
for tag, label in (("", "main: trial-mean baseline (code)"), ("_trial", "variant: per-trial baseline"), ("_alt", "variant: code's windows")):
    for null in ("p_subject", "p_electrode"):
        sig_any = sum(int(((R[s]["erp_enc" + tag][null] < 0.05).to_numpy() | (R[s]["erp_post" + tag][null] < 0.05).to_numpy()).sum()) for s in R)
        where = [(s, R[s]["chans"].label[i]) for s in R for i in np.where((R[s]["erp_enc" + tag][null] < 0.05).to_numpy() | (R[s]["erp_post" + tag][null] < 0.05).to_numpy())[0]]
        print(f"{label}, {null}: electrodes with a significant cluster in either test: {sig_any} of {n_el}"
              + (f" {where}" if null == "p_subject" else ""))
print(f"expected false positives at alpha 0.05 over {n_el} electrodes and two tests, per-electrode null: {0.05 * 2 * n_el:.1f}")


**Result: not reproduced, resting on the baseline of the code.** With the subject-wide null and the
code's baseline, one electrode of 254 (s9, FORA_11, encoding t test, cluster p = 0.036) has a
significant cluster, and the criterion allows none; with the code's windows one electrode does too
(s5, OFG14). Subtracting each trial's own baseline mean leaves no electrode significant under the
subject-wide null, which the criterion would call reproduced. With a per-electrode null 27, 22 and
13 electrodes have a significant cluster (code's baseline, per-trial baseline, code's windows),
against 25.4 expected by chance over two tests, so the effect is no larger than chance.

## power-theta-sustained

> Task-induced power showed sustained theta band (3 to 7 Hz) activity in MTL, PFC and OFC over
> encoding and delay, with transient activity above 16 Hz, and alpha (8 to 13 Hz) desynchronized in
> the MTL.

Theta is the four filters centred at 3.5 to 6.5 Hz; alpha the three centred at 8 to 11.5 Hz. Time
points: the 31 points of encoding and precue and the 19 of the postcue delay, at 50 ms, in the
trial table's windows and, as a variant, the code's.
All 254 analysed electrodes (the claim is about the power before the exclusions).

In [ ]:
TH = [2, 3, 4, 5]
AL = [6, 7, 8]
HI = [i for i, c in enumerate(fcx2.CENTRES) if c > 16]


def zcurve(r, idx, bands, key):
    zs = np.concatenate([r[key]["enc"][idx][:, bands].mean(axis=1),
                         r[key]["post"][idx][:, bands].mean(axis=1)], axis=-1)
    return zs                                                # (electrodes, 50 time points)


def power_table(key):
    rows = []
    for sid, r in R.items():
        c = r["chans"]
        for reg in ("MTL", "PFC", "OFC"):
            idx = np.where(c.region == reg)[0]
            if len(idx) == 0:
                continue
            th = zcurve(r, idx, TH, key)
            al = zcurve(r, idx, AL, key)
            hi = np.concatenate([r[key]["enc"][idx][:, [h for h in HI if h < r[key]["enc"].shape[1]]],
                               r[key]["post"][idx][:, [h for h in HI if h < r[key]["post"].shape[1]]]], axis=-1)
            rows.append({"subject": sid, "region": reg, "n": len(idx),
                       "theta_frac_above": (th.mean(axis=0) > 1.96).mean(), "theta_median_z": np.median(th.mean(axis=0)),
                       "alpha_frac_below": (al.mean(axis=0) < -1.96).mean(), "alpha_median_z": np.median(al.mean(axis=0)),
                       "el_theta_sustained": ((th > 1.96).mean(axis=1) >= 0.5).mean(),
                       "el_alpha_desync": ((al < -1.96).mean(axis=1) >= 0.5).mean(),
                       "el_any_high_above": int((hi > 1.96).any(axis=(1, 2)).sum()),
                       "theta_z_range": (round(th.min(), 1), round(th.max(), 1))})
    return pd.DataFrame(rows)


for key, label in (("power_z_mean", "trial table's windows (main)"), ("power_z_mean_alt", "code's windows (1.5 s later)")):
    pw = power_table(key)
    print(f"=== {label}")
    print(pw.round(3).to_string(index=False))
    for reg in ("MTL", "PFC", "OFC"):
        d = pw[pw.region == reg]
        print(f"{reg}: (a) subjects with region-average theta z > 1.96 at >= half the time points: {(d.theta_frac_above >= 0.5).sum()} of {len(d)}; "
            f"(b) electrodes meeting the rule: {np.sum(d.el_theta_sustained * d.n):.0f} of {d.n.sum()}")
    d = pw[pw.region == "MTL"]
    print(f"MTL alpha: (a) subjects with alpha z < -1.96 at >= half the time points: {(d.alpha_frac_below >= 0.5).sum()} of {len(d)}; "
        f"(b) electrodes {np.sum(d.el_alpha_desync * d.n):.0f} of {d.n.sum()}")
    print("electrodes with any z > 1.96 above 16 Hz:", int(pw.el_any_high_above.sum()), "of", int(pw.n.sum()))

**Result: not reproduced.** In the trial table's windows the region-average theta z is above 1.96
over half the time points in 1 of 10 subjects for MTL, 0 of 9 for PFC and 0 of 9 for OFC, and no
subject shows MTL alpha below -1.96 over half the time points (0 of 10). Per electrode, 3 of 36 MTL,
4 of 116 PFC and 8 of 102 OFC electrodes meet the theta rule. In the code's windows the counts are
0, 0 and 0 subjects (1, 1 and 11 electrodes) and 0 of 10 for alpha. All 254 electrodes have some z
above 1.96 above 16 Hz (239 in the code's windows).

## power-condition-exclusions

> Power did not differ between conditions at any time and frequency in 246 of 254 electrodes; the 8
> that differed (5 PFC and 3 OFC, in 4 of 10 subjects) were excluded from further analysis.

The baseline z is an affine map of power per electrode and frequency, shared by all trials, so the t
and F statistics, and the test, are the same on z and on raw power (readings a and b of the claim
give one result).

In [ ]:
rows = []
for tag in ("", "_alt"):
    for sid, r in R.items():
        c = r["chans"]
        for null in ("p_subject", "p_electrode"):
            sig = (np.minimum(r["pow_enc" + tag][null], r["pow_post" + tag][null]) < 0.05).to_numpy()
            for i in np.where(sig)[0]:
                rows.append({"windows": "code" if tag else "trial table", "null": null, "subject": sid, "label": c.label[i],
                             "region": c.region[i], "p_enc": round(r["pow_enc" + tag][null][i], 3),
                             "p_post": round(r["pow_post" + tag][null][i], 3)})
ex = pd.DataFrame(rows)
print(ex[(ex.windows == "trial table") & (ex.null == "p_subject")].to_string(index=False))
print(ex[(ex.windows == "code") & (ex.null == "p_subject")].to_string(index=False))
for w in ("trial table", "code"):
    for null in ("p_subject", "p_electrode"):
        d = ex[(ex.windows == w) & (ex.null == null)]
        print(f"{w} windows, {null}: {len(d)} electrodes ({d.region.value_counts().to_dict()}) in {d.subject.nunique()} subjects; expected 8 (5 PFC, 3 OFC) in 4")


**Result: not reproduced.** In the trial table's windows the subject-wide null finds 7 electrodes
(2 MTL, 1 PFC, 4 OFC) in 4 subjects (s1, s6, s8, s10), against 8 (5 PFC, 3 OFC) in 4; in the code's
windows it finds 12 (6 PFC, 5 OFC, 1 MTL) in 5 subjects. The per-electrode null finds many more (57
and 69). The 7 are excluded from the analyses that follow, which leaves 34 MTL, 115 PFC and 98 OFC
electrodes (the code's windows exclude their own 12).

## plv-theta-peak

> Phase-locking values over the delays peaked in the theta band within the MTL and between MTL and
> frontal (PFC and OFC) electrodes, with broader synchrony in the alpha range.

Readings: (a) imaginary PLV, |Im(mean over trials of exp(i(phi1 - phi2)))|; (b) the phase-lag index;
and the code's own form (`plvanalysis.m`: the PLV of the imaginary parts of the analytic signals
taken as angles) as a variant. Spectra are averaged over time and over the two delays; a peak is a
filter centred at 3.5 to 6.5 Hz larger than both neighbours.

In [ ]:
cent = fcx2.CENTRES[fcx2.PLV_BANDS]
print("filter centres:", cent.tolist())
rows = []
for tag in ("", "_alt"):
    for sid, r in R.items():
        for g in ("MTL-MTL", "MTL-PFC", "MTL-OFC"):
            if len(r["pairs" + tag][g]) == 0:
                continue
            for rd in ("implv", "pli", "code"):
                spec = (r["plv" + tag]["pre"][g][rd].mean(axis=0) + r["plv" + tag]["post"][g][rd].mean(axis=0)) / 2
                peaks = [cent[i] for i in range(1, len(spec) - 1) if spec[i] > spec[i - 1] and spec[i] > spec[i + 1]]
                rows.append({"windows": "code" if tag else "table", "subject": sid, "group": g, "reading": rd, "pairs": len(r["pairs" + tag][g]),
                             "theta_peak": any(3 <= p <= 7 for p in peaks), "peaks_Hz": peaks,
                             "max_at_Hz": cent[int(np.argmax(spec))], "alpha_over_theta": spec[7:9].mean() / spec[2:6].mean(),
                             "spectrum": np.round(spec / spec.max(), 2).tolist()})
pl = pd.DataFrame(rows)
print(pl[pl.windows == "table"].drop(columns="spectrum").to_string(index=False))
print(pl.groupby(["windows", "reading", "group"]).theta_peak.agg(["sum", "size"]).to_string())
print(pl[(pl.reading == "implv") & (pl.windows == "table")][["subject", "group", "spectrum"]].to_string(index=False))

**Result: reproduced.** In the trial table's windows, the imaginary PLV (reading a) has a local
maximum in 3 to 7 Hz in 4 of 8 subjects for MTL to MTL pairs, 7 of 9 for MTL to PFC and 5 of 7 for
MTL to OFC; the phase-lag index gives 6 of 8, 8 of 9 and 5 of 7. In the code's windows the imaginary
PLV gives 7 of 9, 8 of 9 and 4 of 7. The rule asks for one subject per group, a weak test; the peaks
are often one of several local maxima, and the largest value is in 3 to 7 Hz in fewer subjects
(column max_at_Hz).

## psi-theta-shift

> Theta-band PSI showed frontal-to-MTL leads during the precue delay and shifted toward MTL leads (a
> bidirectional network) after the test cue, for MTL to PFC and MTL to OFC pairs.

Readings of the spectral step: sliding 3-cycle windows, as M5 of the claims file gives the
Methods (main; at 3 Hz the 1 s window runs into the zero padding of the 900 ms segment), the code's
single Hanning window (`mtmfft`, `psi_compute.m`), and the code's own form of PSI (`psianalysis.m`,
per-trial PSI against a null of trial-mean coherency). Readings of M7: the response is the PSI z or
the raw PSI. Each in the trial table's windows and in the code's ("/ code win"). Positive PSI means
the MTL leads. The calibration table gives each form's postcue z with the second electrode's trials
shuffled, which removes any trial-by-trial coupling.

In [ ]:
def pair_table(key, tag=""):
    rows = []
    for sid, r in R.items():
        for g in ("MTL-PFC", "MTL-OFC"):
            for a, b in r["pairs" + tag][g]:
                for w in ("pre", "post"):
                    d = r["psi" + tag][(w, key)]
                    rows.append({"subject": sid, "net": g, "pair": f"{sid}-{a}-{b}", "interval": w,
                                 "z": d["z"][a, b], "raw": d["raw"][a, b]})
    return pd.DataFrame(rows)


def lmm_interval(d, y):
    # Postcue minus precue with random intercepts for subject and pair: with two intervals per
    # pair, the mixed model's F test of interval is the within-pair F test, F = t^2 of the
    # paired differences (fcx2.within_unit_anova).
    if len(d) == 0:
        return np.nan, np.nan, np.nan
    w = d.pivot_table(index="pair", columns="interval", values=y)
    Y = w[["pre", "post"]].to_numpy()
    tab, res_df = fcx2.within_unit_anova(Y)
    diff = (Y[:, 1] - Y[:, 0]).mean()
    return diff, float(tab.loc[0, "F"]), float(tab.loc[0, "p"])


rows = []
for key, tag in [(k, t) for t in ("", "_alt") for k in ("mtmfft", "sliding", "code")]:
    pt = pair_table(key, tag)
    for g in ("MTL-PFC", "MTL-OFC"):
        d = pt[pt.net == g]
        pre = d[d.interval == "pre"]
        post = d[d.interval == "post"]
        for y in ("z", "raw"):
            b, t, p = lmm_interval(d, y)
            rows.append({"reading": key + (" / code win" if tag else ""), "net": g, "pairs": len(pre), "subjects": pre.subject.nunique(), "response": y,
                         "pre_mean_z": pre.z.mean(), "pre_below": int((pre.z < -1.96).sum()), "pre_above": int((pre.z > 1.96).sum()),
                         "post_mean_z": post.z.mean(), "post_below": int((post.z < -1.96).sum()), "post_above": int((post.z > 1.96).sum()),
                         "post_minus_pre": b, "F": t, "p": p})
ps = pd.DataFrame(rows)
print(ps.round(4).to_string(index=False))
print("expected pairs: MTL-PFC 172 +- 17.2, MTL-OFC 303 +- 30.3")
for rd in ps.reading.unique():
    for y in ("z", "raw"):
        d = ps[(ps.reading == rd) & (ps.response == y)]
        p1 = (d.pre_mean_z < 0) & (d.pre_below > d.pre_above)
        p2 = (d.post_minus_pre > 0) & (d.p < 0.05)
        print(f"{rd:16s} {y:3s}: part (1) holds in {int(p1.sum())} of 2 networks, part (2) in {int(p2.sum())} of 2 "
              f"({', '.join(d.net[p2]) or 'none'})")
# calibration: postcue z with the second electrode's trials shuffled against the first's (no
# trial-by-trial coupling left), over the same pairs, against the real postcue z
rows = []
for key in ("mtmfft", "sliding", "code"):
    real = np.concatenate([[r["psi"][("post", key)]["z"][a, b] for g in ("MTL-PFC", "MTL-OFC") for a, b in r["pairs"][g]] for r in R.values()])
    shuf = np.concatenate([[r["psi"][("post", key, "shuf")]["z"][a, b] for g in ("MTL-PFC", "MTL-OFC") for a, b in r["pairs"][g]] for r in R.values()])
    for name, v in (("real", real), ("trials shuffled", shuf)):
        rows.append({"form": key, "data": name, "pairs": len(v), "z_sd": np.std(v), "frac_above_1.96": np.mean(v > 1.96),
                     "frac_below_-1.96": np.mean(v < -1.96)})
print("\ntheta PSI, postcue: calibration (nominal 0.025 in each tail)")
print(pd.DataFrame(rows).round(3).to_string(index=False))

**Result: not reproduced, resting on the window placement.** With sliding windows in the trial
table's windows, neither part holds in either network: precue mean z is 0.25 (MTL-PFC) and 0.01
(MTL-OFC), with fewer pairs below -1.96 than above (45 against 55, 82 against 87), and postcue does
not exceed precue (MTL-PFC raw p = 0.27; MTL-OFC raw PSI falls, p < 0.0001). The code's single
window gives part (2) for MTL-PFC on raw PSI only (p = 0.014, while the z falls), with MTL-OFC raw
PSI falling (p = 0.001), which the criterion would call partial. In the code's windows, sliding
windows give part (1) in both networks and part (2) for MTL-OFC on raw PSI only (p = 0.027), also
partial, and the single window gives part (1) in one network and part (2) in neither. The code's
own form of PSI gives both parts in both networks in the code's windows, and part (1) and part (2)
in one network each in the table's (MTL-OFC postcue above precue, p = 0.0028 on z), but it is not
calibrated: with
trials shuffled 57% of pairs pass z > 1.96 (z SD 44), as many as on the real data (57%). The
sliding-window z is also too wide (SD 2.3 with trials shuffled, 24% above 1.96), which bears on
part (1) only; the single window is near nominal (SD 0.92, 2.2% above). The pair counts are 170 and
289 (expected 172 and 303, within tolerance).

## crossspectral-shift

> Theta phase to amplitude directionality (cross-spectral PSI) was absent in the precue delay and
> shifted to theta phase leading amplitude after the test cue, within MTL, PFC and OFC and in all four
> interregional directions.

Readings: the PSI of the Methods (Nolte et al. 2008) from sliding 3-cycle windows (M5, main) and
from the code's single window, and the code's own form of PSI as a variant; M7 response z or raw;
each in the trial table's windows and in the code's. The calibration table below gives each form's
postcue z with the amplitude electrode's trials shuffled against the phase electrode's.

In [ ]:
def unit_frame(tag):
    units = []
    for sid, r in R.items():
        for i, u in enumerate(r["units" + tag]):
            units.append({"subject": sid, "uid": f"{sid}-{u['ph']}-{u['amp']}", "pair": f"{sid}-{min(u['ph'], u['amp'])}-{max(u['ph'], u['amp'])}",
                          "idx": i, **{k: v for k, v in u.items() if not k.startswith("zf")}})
    return pd.DataFrame(units)


U, U_ALT = unit_frame(""), unit_frame("_alt")
SYSTEMS = ["MTL", "PFC", "OFC", "MTL->PFC", "PFC->MTL", "MTL->OFC", "OFC->MTL"]
rows = []
for key in ["mtmfft", "sliding", "code", "mtmfft / code win", "sliding / code win", "code / code win"]:
    UU = U_ALT if "code win" in key else U
    k0 = key.split(" /")[0]
    for sname in SYSTEMS:
        d = UU[UU.system == sname].rename(columns=lambda c: c.replace(f"_{k0}", f"_{key}") if c.startswith(("z_", "raw_")) and c.endswith(f"_{k0}") else c)
        long = pd.concat([pd.DataFrame({"subject": d.subject, "pair": d.uid, "interval": w,
                                        "z": d[f"z_{w}_{key}"], "raw": d[f"raw_{w}_{key}"]}) for w in ("pre", "post")])
        for y in ("z", "raw"):
            b, t, p = lmm_interval(long, y)
            rows.append({"reading": key, "system": sname, "units": len(d), "subjects": d.subject.nunique(), "response": y,
                         "pre_mean_z": d[f"z_pre_{key}"].mean(), "post_mean_z": d[f"z_post_{key}"].mean(),
                         "pre_above_1.96": int((d[f"z_pre_{key}"] > 1.96).sum()),
                         "pre_frac_above": (d[f"z_pre_{key}"] > 1.96).mean(), "post_minus_pre": b, "F": t, "p": p})
cx = pd.DataFrame(rows)
print(cx.round(4).to_string(index=False))
print("expected electrodes in the models: MTL 36 +- 3.6, PFC 111 +- 11.1, OFC 99 +- 9.9")
for key in ["mtmfft", "sliding", "code", "mtmfft / code win", "sliding / code win", "code / code win"]:
    for y in ("z", "raw"):
        d = cx[(cx.reading == key) & (cx.response == y)]
        print(f"{key}/{y}: systems with post > pre at p < 0.05: {int(((d.post_minus_pre > 0) & (d.p < 0.05)).sum())} of 7; "
              f"systems with at most 5% precue units above 1.96: {int((d['pre_frac_above'] <= 0.05).sum())} of 7")# calibration: postcue z with the amplitude electrode's trials shuffled against the phase
# electrode's, against the real postcue z, every unit (nominal 0.025 above 1.96)
rows = []
for key in ("mtmfft", "sliding", "code"):
    for sname in SYSTEMS:
        d = U[U.system == sname]
        for name, col in (("real", f"z_post_{key}"), ("trials shuffled", f"z_post_{key}_shuf")):
            rows.append({"form": key, "system": sname, "data": name, "units": len(d), "z_sd": d[col].std(),
                         "frac_above_1.96": (d[col] > 1.96).mean()})
cal = pd.DataFrame(rows)
print("\ncross-spectral PSI, postcue: calibration")
print(cal.pivot_table(index=["form", "system"], columns="data", values="frac_above_1.96").round(3).to_string())
print(cal[cal.units > 0].groupby(["form", "data"]).apply(lambda g: pd.Series({"units": g.units.sum(), "z_sd_mean": g.z_sd.mean(),
      "frac_above_1.96": np.average(g["frac_above_1.96"], weights=g.units)})).round(3).to_string())
print("paper's selection rates (counts of phase-directing-counts over the units of crossspectral-shift):",
      {k: round(v / n, 2) for (k, v), n in zip({"MTL": 13, "PFC": 52, "OFC": 38, "MTL->PFC": 56, "PFC->MTL": 85, "MTL->OFC": 93, "OFC->MTL": 115}.items(),
                                                (36, 111, 99, 172, 172, 303, 303))})


**Result: not reproduced.** With the PSI of the Methods and sliding windows, at most 1% of precue
units per system pass z > 1.96 (part 2 holds), but postcue exceeds precue at p < 0.05 in 1 of 7
systems (OFC to MTL) on z and on raw PSI, and MTL to OFC falls. The single window gives 0 of 7 on z
and 1 of 7 on raw PSI; the code's form 3 of 7, with 36% to 47% of precue units above 1.96. In the
code's windows the most any reading gives is 1 (single window), 2 (sliding) and 2 (code's form) of 7. No reading reaches the four systems that partial
needs. The models hold 34 MTL, 115 PFC and 98 OFC electrodes (expected 36, 111 and 99, all within
tolerance). Calibration: with trials shuffled, the code's form passes 57% of units (z SD 23), as
many as on the real data (56%); the Methods' forms pass 0% (single window) and 0.1% (sliding).

## phase-directing-counts

> In the postcue delay, theta phase significantly led higher-frequency amplitude in 13 MTL, 52 PFC
> and 38 OFC electrodes, and in 56 MTL to PFC, 85 PFC to MTL, 93 MTL to OFC and 115 OFC to MTL
> electrode pairs.

In [ ]:
EXP = {"MTL": 13, "PFC": 52, "OFC": 38, "MTL->PFC": 56, "PFC->MTL": 85, "MTL->OFC": 93, "OFC->MTL": 115}
EXP_SUBJ = {"MTL": 8, "PFC": 8, "OFC": 8, "MTL->PFC": 9, "PFC->MTL": 9, "MTL->OFC": 5, "OFC->MTL": 5}
rows = []
for key in ["mtmfft", "sliding", "code", "mtmfft / code win", "sliding / code win", "code / code win"]:
    UU = U_ALT if "code win" in key else U
    k0 = key.split(" /")[0]
    for sname in SYSTEMS:
        d = UU[(UU.system == sname) & (UU[f"z_post_{k0}"] > 1.96)]
        rows.append({"reading": key, "system": sname, "count": len(d), "expected": EXP[sname],
                     "within_10pct": abs(len(d) - EXP[sname]) <= 0.1 * EXP[sname],
                     "subjects": d.subject.nunique(), "expected_subjects": EXP_SUBJ[sname],
                     "per_subject": d.subject.value_counts().to_dict()})
pdc = pd.DataFrame(rows)
print(pdc.to_string(index=False))
print(pdc.groupby("reading").within_10pct.sum().to_string())

**Result: not reproduced.** With the PSI of the Methods and sliding windows the postcue counts are
0 MTL, 0 PFC, 0 OFC, 0 MTL to PFC, 1 PFC to MTL, 3 MTL to OFC and 2 OFC to MTL (0 of 7 within 10%);
the single window selects none. The code's form gives 14, 49, 47, 105, 76, 189 and 170 (expected
13, 52, 38, 56, 85, 93, 115), 2 of 7 within 10%, with MTL electrodes in 5 subjects and MTL to OFC
pairs in 7 (expected 8 and 5); in the code's windows 1 of 7. The code's form passes as many units
with trials shuffled (57%) as on the real data, and its rates (41% to 65% per system) are of the
order of the paper's (31% to 49%), so its counts are not evidence of phase direction.

## crossspectral-equal-strength

> Among phase-directing pairs, the strength of theta phase directionality did not differ across
> amplitude frequencies or between the MTL to frontal and frontal to MTL directions.

**Blocked (methods_underspecified).** The claim is about the phase-directing pairs. The Methods'
PSI selects 0 MTL to PFC pairs and 1 PFC to MTL pair (sliding windows; none with the single
window), too few for the model. The code's form selects 57% of pairs with trials shuffled, against
the 2.5% its z > 1.96 threshold implies, so it does not select phase-directing pairs, and no
selection with a defensible basis is left. The model on the code's selection is shown below as
a variant, not judged. Frequency (16 levels) is within pair, direction between pairs. With random
intercepts and every pair having all 16 frequencies, the mixed model's tests of frequency and
frequency x direction are the within-pair F tests computed by `fcx2.within_unit_anova`; the
direction main effect is tested on the pair means with a subject random intercept.

In [ ]:
ZF = {}
for sid, r in R.items():
    for i, u in enumerate(r["units"]):
        ZF[(sid, i)] = u.get("zf_post_code")


def direction_tests(d, Y, groups):
    tab, res_df = fcx2.within_unit_anova(Y, groups)
    m = pd.DataFrame({"y": Y.reshape(len(Y), -1).mean(axis=1), "dir": groups, "subject": d.subject.to_numpy()})
    md_ = smf.mixedlm("y ~ dir", m, groups=m["subject"]).fit(reml=True)
    return tab, res_df, md_.params["dir"], md_.pvalues["dir"]


for net, (a, b) in {"MTL-PFC": ("MTL->PFC", "PFC->MTL"), "MTL-OFC": ("MTL->OFC", "OFC->MTL")}.items():
    d = U[U.system.isin([a, b]) & (U.z_post_code > 1.96)]
    Y = np.stack([ZF[(s, i)][:16] for s, i in zip(d.subject, d.idx)])
    g = (d.system == b).astype(int).to_numpy()
    tab, res_df, bdir, pdir_ = direction_tests(d, Y, g)
    ss_f, ss_df_ = tab.loc[0, "ss"], tab.loc[1, "ss"]
    res_ss = tab.attrs["res_ss"]
    F_main = (ss_f / 15) / ((res_ss + ss_df_) / (res_df + 15))
    print(f"{net}: {len(d)} pairs ({(g == 0).sum()} {a}, {(g == 1).sum()} {b}) in {d.subject.nunique()} subjects")
    print(f"  (b) with interaction: frequency F(15,{res_df}) = {tab.loc[0, 'F']:.2f}, p = {tab.loc[0, 'p']:.3g}; "
          f"frequency x direction F = {tab.loc[1, 'F']:.2f}, p = {tab.loc[1, 'p']:.3g}")
    print(f"  (a) main effects only: frequency F(15,{res_df + 15}) = {F_main:.2f}, p = {stats.f.sf(F_main, 15, res_df + 15):.3g}")
    print(f"  direction (pair means, subject random intercept): {b} minus {a} = {bdir:.3f}, p = {pdir_:.3g}")
    print("  mean z per frequency:", np.round(Y.mean(axis=0), 2).tolist())
for reg in ("MTL", "PFC", "OFC"):
    d = U[(U.system == reg) & (U.z_post_code > 1.96)]
    if len(d) < 2:
        print(reg, "local variant: fewer than 2 electrodes")
        continue
    Y = np.stack([ZF[(s, i)][:16] for s, i in zip(d.subject, d.idx)])
    tab, res_df = fcx2.within_unit_anova(Y)
    print(f"local variant {reg}: {len(d)} electrodes, frequency F(15,{res_df}) = {tab.loc[0, 'F']:.2f}, p = {tab.loc[0, 'p']:.3g}")

**Result: blocked.** On the pairs the code's form selects (not judged), frequency x direction is
significant in both networks (p = 2e-05 and 0.00019), frequency in MTL-OFC (p = 2e-09) and
direction in both (p = 0.020 and 0.0057).

## theta-peak-bands

> Individual theta peaks were found at 2 to 5 Hz in 3 subjects, 3 to 6 Hz in 2, 4 to 7 Hz in 2 and 5
> to 8 Hz in 3.

The six filters are 1.5-4, 2-5, 3-6, 4-7, 5-8 and 6-10 Hz. A maximum at 1.5-4 or 6-10 Hz has no
3 Hz wide band of the paper's list; it is counted apart, and for PAC the nearest band of the list is
used (2-5 or 5-8 Hz, an assumption).
Reading (b) is shown for the electrodes of the Methods' PSI (single window and sliding windows) and
of the code's form.

In [ ]:
names = {1: "1.5-4", 2: "2-5", 3: "3-6", 4: "4-7", 5: "5-8", 6: "6-10"}
rows = []
for sid, r in R.items():
    rows.append({"subject": sid, "a_all_electrodes": names[r["peak_band_all"]],
                 "b_directing_methods_psi": names.get(r["peak_band_dir"], "none") if r["peak_band_dir"] else "no electrodes",
                 "b_directing_sliding_psi": names.get(r["peak_band_dir_sliding"], "none") if r["peak_band_dir_sliding"] else "no electrodes",
                 "b_directing_code_psi": names.get(r["peak_band_dir_code"], "none") if r["peak_band_dir_code"] else "no electrodes",
                 "z_curve_all": np.round(r["peak_curve_all"], 2).tolist(), "theta_band_used_for_PAC": r.get("theta_band_used")})
pk = pd.DataFrame(rows)
print(pk.to_string(index=False))
exp = {"2-5": 3, "3-6": 2, "4-7": 2, "5-8": 3}
for col in ("a_all_electrodes", "b_directing_methods_psi", "b_directing_sliding_psi", "b_directing_code_psi"):
    vc = pk[col].value_counts().to_dict()
    overlap = sum(min(vc.get(k, 0), v) for k, v in exp.items())
    print(f"{col}: {vc}; all four counts match: {all(vc.get(k, 0) == v for k, v in exp.items())}; overlap with the paper's counts {overlap} of 10")

**Result: not reproduced.** Over all analysed electrodes (reading a) the maxima are at 1.5-4 Hz in 4
subjects, 3-6 Hz in 3, 2-5 Hz in 1, 5-8 Hz in 1 and 6-10 Hz in 1: 5 subjects peak outside the
paper's four bands, and the overlap with the paper's counts is 4 of 10 (partial needs 7). For
reading (b) the Methods' single-window PSI selects no electrodes, its sliding-window PSI selects
electrodes in 3 subjects (overlap 0 of 10), and the code's form gives an overlap of 4 of 10. The
power curves are flat in most subjects (z mostly within -0.9 to 2.2; s9 up to 5.5), so the maxima
are weak.

## PAC (claims pac-mtl-spatial to pac-mtl-ofc-direction)

**All eight PAC claims are blocked (methods_underspecified).** They are about the theta-phase
directing electrodes and pairs. The Methods' PSI selects 0 MTL, 0 PFC, 0 OFC, 0 MTL to PFC, 1 PFC to
MTL, 3 MTL to OFC and 2 OFC to MTL units (sliding windows; none with the single window), too few for
any model, and the code's form, which selects enough, passes as many units with trials shuffled
(57%) as on the real data, against the 2.5% its threshold implies. The models below run on the
code's selection and are shown as a variant, not judged.

PAC z is computed for every unit that is phase-directing under the single-window PSI or the code's
form, in the trial table's postcue window. Readings of the surrogate: (a) amplitudes permuted across
trials (code), (b) amplitude bands shuffled within trials. Models as in M10, computed as within-unit
F tests (`fcx2.within_unit_anova`); a direction main effect on unit means with a subject random
intercept.

In [ ]:
PAC = {}
for sid, r in R.items():
    for p in r["pac"]:
        PAC[(sid, p["unit"], p["cond"], p["reading"])] = p["z"][:16]
AMP_C = fcx2.CENTRES[fcx2.AMP_BANDS][:16]
TBIN = np.arange(19) * 0.05
EFF = {(0,): "condition", (1,): "frequency", (2,): "time", (0, 1): "cond x freq", (0, 2): "cond x time",
       (1, 2): "freq x time", (0, 1, 2): "cond x freq x time"}


def pac_units(system):
    d = U[(U.system == system) & (U.z_post_code > 1.96)]
    return d


def pac_array(d, cond, rd):
    return np.stack([np.stack([PAC[(s, i, 1, rd)], PAC[(s, i, cond, rd)]]) for s, i in zip(d.subject, d.idx)])  # (u, 2, 16, 19)


def pac_model(system, cond, rd):
    d = pac_units(system)
    Y = pac_array(d, cond, rd)
    tab, res_df = fcx2.within_unit_anova(Y)
    diff = (Y[:, 1] - Y[:, 0]).mean(axis=0)                 # (16, 19), relation minus identity
    out = {"system": system, "model": {2: "spatial", 3: "temporal"}[cond], "reading": rd, "units": len(d),
           "subjects": d.subject.nunique(), "res_df": res_df, "mean_diff": diff.mean(),
           "peak_freq": AMP_C[int(np.argmax(diff.mean(axis=1)))], "peak_freq_idx": int(np.argmax(diff.mean(axis=1))),
           "peak_time": TBIN[np.unravel_index(np.argmax(diff), diff.shape)[1]],
           "peak_tf_freq_idx": int(np.unravel_index(np.argmax(diff), diff.shape)[0])}
    for _, row in tab.iterrows():
        out[EFF[row.effect]] = row.p
    return out


rows = []
for system in SYSTEMS:
    for cond in (2, 3):
        for rd in ("a", "b"):
            rows.append(pac_model(system, cond, rd))
PM = pd.DataFrame(rows)
print("Bonferroni threshold 0.05/7 =", round(0.05 / 7, 5))
print(PM.to_string(index=False, float_format=lambda v: f"{v:.3g}"))


def dir_model(a, b, cond, rd):
    da, db = pac_units(a), pac_units(b)
    d = pd.concat([da, db])
    Y = np.concatenate([pac_array(da, cond, rd), pac_array(db, cond, rd)])
    g = np.r_[np.zeros(len(da), int), np.ones(len(db), int)]
    tab, res_df = fcx2.within_unit_anova(Y, g)
    m = pd.DataFrame({"y": Y.mean(axis=(1, 2, 3)), "dir": g, "subject": d.subject.to_numpy()})
    fit = smf.mixedlm("y ~ dir", m, groups=m["subject"]).fit(reml=True)
    out = {"network": f"{a}/{b}", "model": {2: "spatial", 3: "temporal"}[cond], "reading": rd,
           "units": f"{len(da)}+{len(db)}", "direction": fit.pvalues["dir"],
           "mean_a": Y[g == 0].mean(), "mean_b": Y[g == 1].mean(),
           "diff_a": (Y[g == 0][:, 1] - Y[g == 0][:, 0]).mean(), "diff_b": (Y[g == 1][:, 1] - Y[g == 1][:, 0]).mean()}
    for _, row in tab.iterrows():
        e = row.effect
        name = ("direction x " if e[0] == "D" else "") + EFF[tuple(x for x in e if x != "D")]
        out[name] = row.p
    return out


rows = []
for a, b in (("MTL->PFC", "PFC->MTL"), ("MTL->OFC", "OFC->MTL")):
    for cond in (2, 3):
        for rd in ("a", "b"):
            rows.append(dir_model(a, b, cond, rd))
DM = pd.DataFrame(rows)
print("\nDirection models, Bonferroni threshold 0.05/15 =", round(0.05 / 15, 5))
print(DM.T.to_string(float_format=lambda v: f"{v:.3g}"))

In [ ]:
B7, B15 = 0.05 / 7, 0.05 / 15
CONDFX = ["condition", "cond x freq", "cond x time", "cond x freq x time"]
ALLFX = list(EFF.values())


def sel(system, model, rd):
    return PM[(PM.system == system) & (PM.model == model) & (PM.reading == rd)].iloc[0]


verdict = {}
for rd in ("a", "b"):
    s, t = sel("MTL", "spatial", rd), sel("MTL", "temporal", rd)
    p1 = s["condition"] < B7 and s["cond x freq"] < B7 and s.mean_diff > 0
    p2 = abs(s.peak_freq_idx - 5) <= 1 or abs(s.peak_freq_idx - 11) <= 1
    p3 = all(t[f] >= B7 for f in CONDFX)
    verdict[("pac-mtl-spatial", rd)] = (p1, p2, p3)
    verdict[("pac-frontal-no-condition", rd)] = all(sel(sy, mo, rd)[f] >= B7 for sy in ("PFC", "OFC") for mo in ("spatial", "temporal") for f in CONDFX)
    s = sel("MTL->PFC", "spatial", rd)
    q1 = all(s[f] < B7 for f in CONDFX) and s.mean_diff > 0
    q2 = 1 <= s.peak_tf_freq_idx <= 4 and (0.1 <= s.peak_time <= 0.25 or 0.8 <= s.peak_time <= 0.9)
    verdict[("pac-mtl-pfc-spatial", rd)] = (q1, q2, s["condition"] < B7 and s.mean_diff > 0)
    t = sel("MTL->PFC", "temporal", rd)
    verdict[("pac-mtl-pfc-temporal", rd)] = t["condition"] < B7 and t.mean_diff > 0
    verdict[("pac-pfc-mtl-no-condition", rd)] = all(sel("PFC->MTL", mo, rd)[f] >= B7 for mo in ("spatial", "temporal") for f in CONDFX)
    s, t = sel("MTL->OFC", "spatial", rd), sel("MTL->OFC", "temporal", rd)
    r1 = s["condition"] < B7 and s["cond x freq"] < B7 and s.mean_diff > 0
    r2 = s.peak_freq_idx <= 1
    r3 = all(t[f] >= B7 for f in CONDFX) and all(sel("OFC->MTL", mo, rd)[f] >= B7 for mo in ("spatial", "temporal") for f in CONDFX)
    verdict[("pac-mtl-ofc-spatial", rd)] = (r1, r2, r3)
    for net, key in (("MTL->PFC/PFC->MTL", "pac-mtl-pfc-direction"), ("MTL->OFC/OFC->MTL", "pac-mtl-ofc-direction")):
        out = []
        for mo in ("spatial", "temporal"):
            d = DM[(DM.network == net) & (DM.model == mo) & (DM.reading == rd)].iloc[0]
            out.append({"model": mo, "cond_x_dir": d["direction x condition"], "cond_x_dir_larger_MTL_first": d.diff_a > d.diff_b,
                        "direction": d["direction"], "higher": "first (MTL->)" if d.mean_a > d.mean_b else "second (->MTL)",
                        "any_effect_below": any(d[c] < B15 for c in d.index if c not in ("network", "model", "reading", "units", "mean_a", "mean_b", "diff_a", "diff_b"))})
        verdict[(key, rd)] = out
for k, v in verdict.items():
    print(k, v)

**Result: blocked, all eight claims.** On the code's selection (not judged) none of the eight
patterns appears under either surrogate reading: the MTL spatial condition effect is p = 0.56 (a),
the frontal and PFC to MTL models have condition effects below 0.05/7 (PFC spatial condition x
frequency p = 8.3e-05, PFC to MTL spatial condition p = 0.0004), the MTL to PFC condition effects are
p = 0.23 (spatial) and 0.61 (temporal), the MTL to OFC spatial condition effect is p = 0.18, and the
direction models have effects below 0.05/15 in every network and model.

## Summary

Outcomes are in the attempt record: 5 claims reproduced (behaviour, electrode and trial counts, PLV
peak), 7 not reproduced and 9 blocked. The analysis follows the claims file and, for details it
lacks, the paper's Methods and the analysis code deposited with the data. The main differences from
the paper's method: the windows are placed by the trial table, 1.5 s earlier on the derived time
axes than the deposited code places them, because the 900 and 1150 ms delay trials part where the
table puts the delay-dependent events and not where the code does; the code's windows are run as a
variant. The phase-directing units on which crossspectral-equal-strength and the eight PAC claims
rest cannot be selected: the Methods' PSI selects at most 3 per system, and the code's form selects
as many units with trials shuffled as on the real data. The behaviour claim is judged with
condition as one numeric term, as the amended claims file says, on the response times of
`summary_file.csv`, whose condition means are the paper's; the trial tables' response times differ
in 7 of 10 subjects, no summary of them reproduces the file's values, and they would show a
condition effect. The mixed models use random intercepts.

What was found about the data set: the trial table and the derived time axes disagree with the
deposited code's windows by 1.5 s; the response times of `summary_file.csv` do not follow from the
trial tables in 7 of 10 subjects; s10's trial table is not on the derived sample grid; s8 and s9 are
at 512 Hz, not the 1000 Hz the descriptor gives; and the deposited PSI and PAC scripts depart from
the Methods (PSI without the conjugate and with a null of another statistic; PAC scripts that cannot
run as given).

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

In [ ]:
print(f"total run time {(time.time() - started) / 60:.1f} min")